# Invariant Computation Time Benchmark

5가지 topological invariant의 계산 시간을 측정합니다:
- **Ord PI**: Ordinary Persistence Image
- **Inter PI**: Interaction Persistence Image
- **3D PI**: 3D Persistence Image
- **Sixpack Rips**: Six-pack (Rips complex)
- **Sixpack Chroma**: Six-pack (chromatic_tda)

## Method
- Stratified random sampling: 12 ground truth classes에서 각 3개 샘플
- 각 샘플당 3회 반복, median 사용
- `time.perf_counter()` 고해상도 wall-clock 측정

In [ ]:
# Install dependencies (Colab)
!pip install gudhi persim chromatic_tda -q

In [ ]:
import os, time, random, warnings
import numpy as np
from collections import defaultdict
from typing import Dict, List
from scipy.stats import norm
import pandas as pd

from gudhi import RipsComplex
from persim import PersistenceImager
import persim.images_weights as weights

warnings.filterwarnings('ignore')

try:
    import chromatic_tda as chro
    HAS_CHROMA = True
    print('chromatic_tda loaded')
except ImportError:
    HAS_CHROMA = False
    print('chromatic_tda NOT available - Sixpack_Chroma will be skipped')

# === Configuration ===
# Google Drive에서 데이터를 사용하려면 아래 경로를 수정하세요
# Colab에서 Drive mount 후 사용:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DATA_DIR = '/content/drive/MyDrive/...'

BASE_DATA_DIR = 'Data/ParamSweep_Input'  # 로컬 경로
A_VALS = [0.00, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAM_LIST = [(x1, x2, x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]

SAMPLES_PER_CLASS = 3
ITERATIONS = 3
RANDOM_SEED = 42
MAX_EDGE = 10  # Rips max edge length

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'Config: {SAMPLES_PER_CLASS} samples/class, {ITERATIONS} iterations, seed={RANDOM_SEED}')
print(f'PARAM_LIST: {len(PARAM_LIST)} parameter sets')

In [ ]:
# Ground Truth Matrix (12 classes)
M1=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3]]
M2=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,4],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,4,4],[2,2,3,3,3,3,3,3]]
M3=[[6,6,7,7,7,7,7,7],[6,6,6,7,7,7,7,7],[9,6,3,3,3,3,3,3],[9,10,3,4,4,3,3,4],[9,10,3,3,4,4,3,4],[9,10,3,4,4,4,4,4],[9,10,3,4,3,4,4,4],[9,10,3,4,3,4,4,4]]
M4=[[6,6,12,12,7,7,7,7],[6,6,12,12,7,7,7,7],[9,6,6,11,7,7,4,4],[9,9,6,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,4,4,4,4,4],[9,9,10,4,4,4,4,4]]
M5=[[6,6,12,12,12,12,7,7],[6,6,12,12,12,12,12,7],[9,9,6,11,11,11,12,11],[9,9,6,11,11,11,4,4],[9,9,13,13,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,10,10,4,4,4,4]]
M6=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,11],[9,9,6,11,11,11,11,11],[9,9,6,6,6,13,4,4],[9,9,6,13,13,4,4,4],[9,9,6,13,4,4,4,4],[9,9,6,13,4,4,4,4]]
M7=[[6,6,12,12,12,12,12,12],[9,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,12],[9,6,6,11,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,11,11,11,4],[9,9,6,6,13,13,4,4],[9,9,6,13,13,4,4,4]]
M8=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,6,11,11,11,11],[9,6,6,6,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,6,11,11,11],[9,9,6,6,13,13,11,11],[9,9,6,6,13,13,11,4]]
GROUND_TRUTH_M = np.asarray([M1,M2,M3,M4,M5,M6,M7,M8])

def get_label(task_id):
    idx = task_id - 1
    RR_idx = idx // 64
    RG_idx = (idx % 64) // 8
    GG_idx = idx % 8
    return GROUND_TRUTH_M[RG_idx][RR_idx][GG_idx]

# Build class -> sample_ids mapping
class_to_ids = defaultdict(list)
for tid in range(1, 513):
    label = get_label(tid)
    class_to_ids[label].append(tid)

classes = sorted(class_to_ids.keys())
print(f'Classes: {classes} ({len(classes)} classes)')
for c in classes:
    print(f'  Class {c}: {len(class_to_ids[c])} samples')

In [ ]:
def load_sample(task_id):
    params = PARAM_LIST[task_id - 1]
    folder = os.path.join(BASE_DATA_DIR, f'ParamSweep_{task_id}_Output')
    pos_file = os.path.join(
        folder, f'Pos_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat'
    )
    types_file = os.path.join(
        folder, f'Types_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat'
    )

    if not os.path.exists(pos_file) or not os.path.exists(types_file):
        raise FileNotFoundError(f'데이터 파일 없음: {folder}')

    positions = np.loadtxt(pos_file, delimiter=',')
    types = np.loadtxt(types_file, dtype=int)

    A = positions[types == 1]
    B = positions[types == 2]
    return A, B

# Stratified sampling
selected_samples = []
for c in classes:
    ids = class_to_ids[c]
    k = min(SAMPLES_PER_CLASS, len(ids))
    chosen = random.sample(ids, k)
    for sid in chosen:
        selected_samples.append((sid, c))

print(f'Total selected samples: {len(selected_samples)}')
for sid, c in selected_samples[:5]:
    print(f'  Sample {sid} -> Class {c}')
print('  ...')

In [ ]:
# === Common Functions ===

def compute_Rips(points, max_edge=10, max_dim=2):
    rips = RipsComplex(points=points, max_edge_length=max_edge)
    st = rips.create_simplex_tree(max_dimension=max_dim)
    return st

def compute_Persistence_barcode(A, max_edge=10):
    fil_A = compute_Rips(A, max_edge)
    fil_A.persistence()
    bar_a0 = fil_A.persistence_intervals_in_dimension(0)
    bar_A0 = [bar for bar in bar_a0 if bar[1] != np.inf]
    bar_a1 = fil_A.persistence_intervals_in_dimension(1)
    bar_A1 = [bar for bar in bar_a1 if bar[1] != np.inf]
    bar0 = [[bar[0], bar[1]] for bar in bar_A0 if bar[1]-bar[0] > 1e-5]
    bar1 = [[bar[0], bar[1]] for bar in bar_A1 if bar[1]-bar[0] > 1e-5]
    return {0: np.array(bar0) if bar0 else np.zeros((0,2)),
            1: np.array(bar1) if bar1 else np.zeros((0,2))}

def compute_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05, normalization=False):
    for key in barcodes:
        if len(barcodes[key]) == 0:
            barcodes[key] = np.zeros((0, 2))
    vector = {}

    pers_imager_h0 = PersistenceImager()
    pers_imager_h0.pixel_size = px_res
    pers_imager_h0.birth_range = (0, 1)
    pers_imager_h0.pers_range = (0, max_eps)
    pers_imager_h0.weight = weights.persistence
    pers_imager_h0.weight_params = {'n': 1}
    pers_imager_h0.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h0 = np.array(barcodes[0])
    if len(bars_h0) > 0:
        img_h0 = pers_imager_h0.transform(bars_h0, skew=False)
    else:
        img_h0 = np.zeros((int(1/px_res), int(max_eps/px_res)))
    img0_1d = np.mean(img_h0, axis=0)

    pers_imager_h1 = PersistenceImager()
    pers_imager_h1.pixel_size = px_res
    pers_imager_h1.birth_range = (0, max_eps)
    pers_imager_h1.pers_range = (0, max_eps / 2)
    pers_imager_h1.weight = weights.persistence
    pers_imager_h1.weight_params = {'n': 1}
    pers_imager_h1.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h1 = np.array(barcodes[1])
    if len(bars_h1) > 0:
        img_h1 = pers_imager_h1.transform(bars_h1, skew=True)
    else:
        img_h1 = np.zeros((int(max_eps/px_res), int((max_eps/2)/px_res)))

    if normalization:
        vector[0] = img0_1d / np.max(img0_1d) if np.max(img0_1d) > 0 else img0_1d
        vector[1] = img_h1.flatten() / np.max(img_h1) if np.max(img_h1) > 0 else img_h1.flatten()
    else:
        vector[0] = img0_1d
        vector[1] = img_h1.flatten()
    return vector

print('Common functions loaded.')

In [ ]:
# === Ord PI Pipeline ===

def run_ord_pi(A, B):
    bar_A = compute_Persistence_barcode(A)
    bar_B = compute_Persistence_barcode(B)
    PI_A = compute_PIs(bar_A)
    PI_B = compute_PIs(bar_B)
    return np.concatenate([PI_A[0], PI_A[1], PI_B[0], PI_B[1]])

print('Ord PI pipeline loaded.')

In [ ]:
# === Mixup Barcode / Inter PI / 3D PI Pipeline ===

def extract_filtration(st):
    pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    simplices = [p[0] for p in pairs]
    filt_values = [p[1] for p in pairs]
    return simplices, filt_values

def _reduce_matrix(columns, n):
    R = [set(col) for col in columns]
    low = [-1] * n
    pivot_to_col = {}
    for j in range(n):
        while R[j]:
            pivot = max(R[j])
            if pivot in pivot_to_col:
                R[j] ^= R[pivot_to_col[pivot]]
            else:
                pivot_to_col[pivot] = j
                low[j] = pivot
                break
        else:
            low[j] = -1
    return R, low

def compute_mixup_barcode(A, B, max_edge=10, max_dim=1):
    total = np.concatenate([A, B], axis=0)
    a = len(A)
    st = compute_Rips(total, max_edge, max_dim=max_dim+1)
    simplices, filt = extract_filtration(st)
    n = len(simplices)
    simplex_dims = [len(s) - 1 for s in simplices]
    sf_to_idx = {s: i for i, s in enumerate(simplices)}
    in_L = [all(v < a for v in s) for s in simplices]
    idx_L = [i for i, b in enumerate(in_L) if b]
    idx_KmL = [i for i, b in enumerate(in_L) if not b]

    BK_original = []
    for s in simplices:
        if len(s) <= 1:
            BK_original.append(set())
        else:
            rows = set()
            for j in range(len(s)):
                face = s[:j] + s[j+1:]
                if face in sf_to_idx:
                    rows.add(sf_to_idx[face])
            BK_original.append(rows)

    row_order = idx_L + idx_KmL
    old_to_new_row = {old: new for new, old in enumerate(row_order)}
    n_L = len(idx_L)

    BK = []
    for col_idx in range(n):
        new_rows = {old_to_new_row[r] for r in BK_original[col_idx]}
        BK.append(new_rows)

    BL = []
    for col_idx in range(n):
        if in_L[col_idx]:
            new_rows = {r for r in BK[col_idx] if r < n_L}
            BL.append(new_rows)
        else:
            BL.append(set())

    RL, lowL = _reduce_matrix(BL, n)
    RK, lowK = _reduce_matrix(BK, n)

    pivotL_to_col = {}
    for j in idx_L:
        if lowL[j] != -1:
            pivotL_to_col[lowL[j]] = j

    pivotK_to_col = {}
    for j in range(n):
        if lowK[j] != -1:
            pivotK_to_col[lowK[j]] = j

    mixup_triples = defaultdict(list)
    for sigma in idx_L:
        if RL[sigma]:
            continue
        dim = simplex_dims[sigma]
        if dim > max_dim:
            continue
        sigma_row = old_to_new_row[sigma]
        birth = filt[sigma]

        tau = pivotL_to_col.get(sigma_row, None)
        death = filt[tau] if tau is not None else np.inf

        tau_prime = pivotK_to_col.get(sigma_row, None)
        death_prime = filt[tau_prime] if tau_prime is not None else np.inf

        if not np.isinf(death) and (np.isinf(death_prime) or death_prime > death):
            death_prime = death

        if np.isinf(death) or abs(death - birth) > 1e-10:
            mixup_triples[dim].append((birth, death_prime, death))

    result = {}
    for dim in sorted(mixup_triples.keys()):
        triples = mixup_triples[dim]
        if triples:
            arr = np.array(triples)
            order = np.argsort(arr[:, 0])
            result[dim] = arr[order]
        else:
            result[dim] = np.empty((0, 3))
    for d in range(max_dim + 1):
        if d not in result:
            result[d] = np.empty((0, 3))
    return result

def compute_Interaction_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05, normalization=False):
    vector = {}
    def make_mixup_weight(mw):
        def weight(birth, persistence, **kwargs):
            return mw
        return weight

    pers_imager_h0 = PersistenceImager()
    pers_imager_h0.pixel_size = px_res
    pers_imager_h0.birth_range = (0, 0.01)
    pers_imager_h0.pers_range = (0, max_eps)
    pers_imager_h0.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h0 = np.asarray(barcodes.get(0, np.zeros((0, 3))))
    if len(bars_h0) > 0:
        b = bars_h0[:, 0]; d_prime = bars_h0[:, 1]; d = bars_h0[:, 2]
        mask = np.isfinite(b) & np.isfinite(d_prime) & np.isfinite(d)
        b, d_prime, d = b[mask], d_prime[mask], d[mask]
        if len(b) > 0:
            bars_for_pi = np.stack([b, d], axis=1)
            mixup_weights = d - d_prime
            pers_imager_h0.weight = make_mixup_weight(mixup_weights)
            img_h0 = pers_imager_h0.transform(bars_for_pi, skew=True)
        else:
            img_h0 = np.zeros((int(1 / px_res), int(max_eps / px_res)))
    else:
        img_h0 = np.zeros((int(1 / px_res), int(max_eps / px_res)))
    vector[0] = np.mean(img_h0, axis=0)

    pers_imager_h1 = PersistenceImager()
    pers_imager_h1.pixel_size = px_res
    pers_imager_h1.birth_range = (0, max_eps)
    pers_imager_h1.pers_range = (0, max_eps / 2)
    pers_imager_h1.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h1 = np.asarray(barcodes.get(1, np.zeros((0, 3))))
    if len(bars_h1) > 0:
        b = bars_h1[:, 0]; d_prime = bars_h1[:, 1]; d = bars_h1[:, 2]
        mask = np.isfinite(b) & np.isfinite(d_prime) & np.isfinite(d)
        b, d_prime, d = b[mask], d_prime[mask], d[mask]
        if len(b) > 0:
            bars_for_pi = np.stack([b, d], axis=1)
            mixup_weights = d - d_prime
            pers_imager_h1.weight = make_mixup_weight(mixup_weights)
            img_h1 = pers_imager_h1.transform(bars_for_pi, skew=True)
        else:
            img_h1 = np.zeros((int(max_eps / px_res), int((max_eps / 2) / px_res)))
    else:
        img_h1 = np.zeros((int(max_eps / px_res), int((max_eps / 2) / px_res)))

    if normalization:
        vector[0] = vector[0] / np.max(vector[0]) if np.max(vector[0]) > 0 else vector[0]
        vector[1] = img_h1.flatten() / np.max(img_h1) if np.max(img_h1) > 0 else img_h1.flatten()
    else:
        vector[1] = img_h1.flatten()
    return vector

def mixup_barcode_translation(mixup_barcode):
    translation = {}
    for i in range(2):
        translated = []
        for bar in mixup_barcode[i]:
            translated.append([bar[0], bar[1] - bar[0], bar[2] - bar[0]])
        translation[i] = translated
    return translation

def compute_3d_persistence_image(mixup_barcodes, resolution=20,
                                  ranges=((0,10),(0,10),(0,10)),
                                  bandwidth=0.4, weight_func=None,
                                  normalization=False):
    translated = mixup_barcode_translation(mixup_barcodes)
    persistence_vectors = {}
    if weight_func is None:
        weight_func = lambda p: 1.0

    x_grid = np.linspace(ranges[0][0], ranges[0][1], resolution)
    y_grid = np.linspace(ranges[1][0], ranges[1][1], resolution)
    z_grid = np.linspace(ranges[2][0], ranges[2][1], resolution)

    for i in range(2):
        barcode = translated[i]
        if i == 0:
            pi = np.zeros((resolution, resolution))
            for point in barcode:
                w = weight_func(point)
                gy = norm.pdf(y_grid, loc=point[1], scale=bandwidth)
                gz = norm.pdf(z_grid, loc=point[2], scale=bandwidth)
                pi += w * np.outer(gy, gz)
        else:
            pi = np.zeros((resolution, resolution, resolution))
            for point in barcode:
                w = weight_func(point)
                gx = norm.pdf(x_grid, loc=point[0], scale=bandwidth)
                gy = norm.pdf(y_grid, loc=point[1], scale=bandwidth)
                gz = norm.pdf(z_grid, loc=point[2], scale=bandwidth)
                pi += w * np.einsum('i,j,k->ijk', gx, gy, gz)
        if normalization:
            mx = np.max(pi)
            if mx > 0:
                pi = pi / mx
        persistence_vectors[i] = pi.flatten()
    return persistence_vectors

def run_inter_pi(A, B):
    mb_ab = compute_mixup_barcode(A, B)
    mb_ba = compute_mixup_barcode(B, A)
    pi_ab = compute_Interaction_PIs(mb_ab)
    pi_ba = compute_Interaction_PIs(mb_ba)
    return np.concatenate([pi_ab[0], pi_ab[1], pi_ba[0], pi_ba[1]])

def run_3d_pi(A, B):
    mb_ab = compute_mixup_barcode(A, B)
    mb_ba = compute_mixup_barcode(B, A)
    pi_ab = compute_3d_persistence_image(mb_ab)
    pi_ba = compute_3d_persistence_image(mb_ba)
    return np.concatenate([pi_ab[0], pi_ab[1], pi_ba[0], pi_ba[1]])

print('Mixup/Inter PI/3D PI pipeline loaded.')

In [ ]:
# === Sixpack Rips Pipeline ===

def divide_filtration(st):
    simplex_filt_pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    list_simplex = [pair[0] for pair in simplex_filt_pairs]
    list_filt = [pair[1] for pair in simplex_filt_pairs]
    return list_simplex, list_filt

def _build_boundary(simplices):
    sf_to_idx = {s: i for i, s in enumerate(simplices)}
    boundary = []
    for s in simplices:
        if len(s) <= 1:
            boundary.append(set())
        else:
            rows = set()
            for j in range(len(s)):
                face = s[:j] + s[j+1:]
                if face in sf_to_idx:
                    rows.add(sf_to_idx[face])
            boundary.append(rows)
    return boundary

def _reduce_with_V(columns):
    m = len(columns)
    R = [set(col) for col in columns]
    V = [{i} for i in range(m)]
    low = [-1] * m
    pivot_of_row = {}
    for i in range(m):
        while R[i]:
            li = max(R[i])
            if li in pivot_of_row:
                owner = pivot_of_row[li]
                R[i] ^= R[owner]
                V[i] ^= V[owner]
            else:
                pivot_of_row[li] = i
                low[i] = li
                break
        else:
            low[i] = -1
    return R, low, V

def compute_all_barcodes(A, B, max_edge=10):
    total = np.concatenate([A, B], axis=0)
    a = len(A)
    st = compute_Rips(total, max_edge=max_edge)
    simplices, filt = divide_filtration(st)
    m = len(simplices)
    in_L = [all(v < a for v in s) for s in simplices]
    idx_L = [i for i, b in enumerate(in_L) if b]
    idx_KmL = [i for i, b in enumerate(in_L) if not b]
    set_idx_KmL = set(idx_KmL)
    g2L = {g: pos for pos, g in enumerate(idx_L)}

    Df = _build_boundary(simplices)
    Rf, lowf, Vf = _reduce_with_V(Df)

    boundary_L = []
    for g_idx in idx_L:
        col = {g2L[r] for r in Df[g_idx] if r in g2L}
        boundary_L.append(col)
    Rg, lowg, Vg = _reduce_with_V(boundary_L)

    row_order = idx_L + idx_KmL
    row_remap = {g: i for i, g in enumerate(row_order)}
    inv_row_remap = {i: g for g, i in row_remap.items()}

    Dim = [{row_remap[r] for r in Df[col_idx]} for col_idx in range(m)]
    Rim, lowim, _ = _reduce_with_V(Dim)
    Vim = [{row_remap[r] for r in Vf[col_idx]} for col_idx in range(m)]

    cycle_cols = [i for i in range(m) if not Rim[i]]
    Dker = [Vim[c] for c in cycle_cols]
    if Dker:
        _, lowker, _ = _reduce_with_V(Dker)
    else:
        lowker = []
    cycle_pos = {c: pos for pos, c in enumerate(cycle_cols)}

    Dcok = []
    for i in range(m):
        if in_L[i]:
            jL = g2L[i]
            if not Rg[jL]:
                Dcok.append({idx_L[pos] for pos in Vg[jL]})
                continue
        Dcok.append(set(Df[i]))
    _, lowcok, _ = _reduce_with_V(Dcok)

    def _format_output(bars_dict):
        out = {}
        for p in [0, 1]:
            if p in bars_dict and bars_dict[p]:
                arr = np.array(bars_dict[p])
                out[p] = arr[np.lexsort((arr[:, 1], arr[:, 0]))]
            else:
                out[p] = np.empty((0, 2))
        return out

    image_bars = defaultdict(list)
    for tau in range(m):
        if not Rf[tau] or lowim[tau] == -1:
            continue
        sigma = inv_row_remap[lowim[tau]]
        if sigma in g2L:
            bv, dv = filt[sigma], filt[tau]
            if bv != dv:
                p = len(simplices[sigma]) - 1
                if p >= 0:
                    image_bars[p].append((bv, dv))

    kernel_bars = defaultdict(list)
    for tau in idx_L:
        jL = g2L[tau]
        if not Rg[jL] or Rf[tau] or tau not in cycle_pos:
            continue
        local_col = cycle_pos[tau]
        if local_col >= len(lowker):
            continue
        low_local = lowker[local_col]
        if low_local == -1:
            continue
        sigma = inv_row_remap[low_local]
        if in_L[sigma]:
            continue
        bv, dv = filt[sigma], filt[tau]
        if bv != dv:
            p = len(simplices[sigma]) - 2
            if p >= 0:
                kernel_bars[p].append((bv, dv))

    cok_bars = defaultdict(list)
    for tau in range(m):
        if not Rf[tau] or lowim[tau] == -1:
            continue
        row_global = inv_row_remap[lowim[tau]]
        if row_global not in set_idx_KmL:
            continue
        lowc = lowcok[tau]
        if lowc == -1:
            continue
        sigma = lowc
        bv, dv = filt[sigma], filt[tau]
        if bv != dv:
            p = len(simplices[sigma]) - 1
            if p >= 0:
                cok_bars[p].append((bv, dv))

    return {
        'image': _format_output(image_bars),
        'kernel': _format_output(kernel_bars),
        'cokernel': _format_output(cok_bars)
    }

def run_sixpack_rips(A, B):
    total = np.concatenate([A, B], axis=0)
    sp_ab = compute_all_barcodes(A, B)
    sp_ba = compute_all_barcodes(B, A)
    PB_total = compute_Persistence_barcode(total)
    PB_A = compute_Persistence_barcode(A)
    PB_B = compute_Persistence_barcode(B)

    sp_ab.update({'complex': PB_total, 'sub_complex': PB_A, 'relative': PB_B})
    sp_ba.update({'complex': PB_total, 'sub_complex': PB_B, 'relative': PB_A})

    vectors = []
    for sp in [sp_ab, sp_ba]:
        for key in sorted(sp.keys()):
            pi = compute_PIs(sp[key])
            vectors.extend([pi[0], pi[1]])
    return np.concatenate(vectors)

print('Sixpack Rips pipeline loaded.')

In [ ]:
# === Sixpack Chroma Pipeline ===

def compute_PIs_chroma(barcodes, max_eps=10, px_res=0.1, sigma=0.025, normalization=False):
    if 0 not in barcodes:
        barcodes[0] = np.zeros((0, 2))
    if 1 not in barcodes:
        barcodes[1] = np.zeros((0, 2))
    for key in barcodes:
        if len(barcodes[key]) == 0:
            barcodes[key] = np.zeros((0, 2))
    vector = {}

    pers_imager_h0 = PersistenceImager()
    pers_imager_h0.pixel_size = px_res
    pers_imager_h0.birth_range = (0, 0.01)
    pers_imager_h0.pers_range = (0, max_eps)
    pers_imager_h0.weight = weights.persistence
    pers_imager_h0.weight_params = {'n': 1}
    pers_imager_h0.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h0 = np.array(barcodes.get(0, np.zeros((0, 2))))
    if len(bars_h0) > 0:
        img_h0 = pers_imager_h0.transform(bars_h0, skew=True)
    else:
        img_h0 = np.zeros((int(1/px_res), int(max_eps/px_res)))
    vector[0] = np.mean(img_h0, axis=0)

    pers_imager_h1 = PersistenceImager()
    pers_imager_h1.pixel_size = px_res
    pers_imager_h1.birth_range = (0, max_eps)
    pers_imager_h1.pers_range = (0, max_eps / 2)
    pers_imager_h1.weight = weights.persistence
    pers_imager_h1.weight_params = {'n': 1}
    pers_imager_h1.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h1 = np.array(barcodes.get(1, np.zeros((0, 2))))
    if len(bars_h1) > 0:
        img_h1 = pers_imager_h1.transform(bars_h1, skew=True)
    else:
        img_h1 = np.zeros((int(max_eps/px_res), int((max_eps/2)/px_res)))

    if normalization:
        vector[0] = vector[0] / np.max(vector[0]) if np.max(vector[0]) > 0 else vector[0]
        vector[1] = img_h1.flatten() / np.max(img_h1) if np.max(img_h1) > 0 else img_h1.flatten()
    else:
        vector[1] = img_h1.flatten()
    return vector

if HAS_CHROMA:
    def convert_into_diagram(diagram):
        diagrams = {}
        for dim, pairs in diagram.items():
            filtered = [(float(b), float(d)) for (b, d) in pairs if np.isfinite(d)]
            diagrams[dim] = np.array(filtered) if filtered else np.zeros((0,2))
        return diagrams

    def convert_six_pack_to_diagram(six_pack):
        packs = {}
        for key, dgm in six_pack.items():
            packs[key] = convert_into_diagram(dgm)
        return packs

    def compute_six_pack_diagrams(points, labels, max_edge=10):
        chro_alpha = chro.ChromaticAlphaComplex(points, labels, max_alpha=max_edge)
        sc = chro_alpha.get_simplicial_complex(
            sub_complex='0', full_complex='all', relative=None)
        six_pack = sc.bars_six_pack()
        return convert_six_pack_to_diagram(six_pack)

    def run_sixpack_chroma(A, B):
        points = np.array(np.concatenate([A, B], axis=0))
        labels = np.array(np.concatenate([np.zeros(len(A)), np.ones(len(B))]))
        sp_ab = compute_six_pack_diagrams(points, labels)
        sp_ba = compute_six_pack_diagrams(points, 1-labels)
        vectors = []
        for sp in [sp_ab, sp_ba]:
            for key in sorted(sp.keys()):
                pi = compute_PIs_chroma(sp[key])
                vectors.extend([pi[0], pi[1]])
        return np.concatenate(vectors)

    print('Sixpack Chroma pipeline loaded.')
else:
    print('Sixpack Chroma pipeline SKIPPED (no chromatic_tda).')

In [ ]:
# === Benchmark Execution ===
import gc

invariants = {
    'Ord_PI': run_ord_pi,
    'Inter_PI': run_inter_pi,
    '3D_PI': run_3d_pi,
    'Sixpack_Rips': run_sixpack_rips,
}
if HAS_CHROMA:
    invariants['Sixpack_Chroma'] = run_sixpack_chroma

results = []  # list of dicts

total_samples = len(selected_samples)
for sample_idx, (task_id, gt_class) in enumerate(selected_samples):
    print(f'\n=== Sample {sample_idx+1}/{total_samples}: task_id={task_id}, class={gt_class} ===')

    try:
        A, B = load_sample(task_id)
        print(f'  Loaded: A={A.shape}, B={B.shape}')
    except Exception as e:
        print(f'  ERROR loading sample: {e}')
        continue

    for inv_name, inv_func in invariants.items():
        times = []
        success = True
        for it in range(ITERATIONS):
            try:
                # 다음 계산 전에 메모리를 명시적으로 확보하여 공정한 환경 만들기
                gc.collect()

                t0 = time.perf_counter()
                _ = inv_func(A.copy(), B.copy())
                t1 = time.perf_counter()
                elapsed = t1 - t0
                times.append(elapsed)
                print(f'  {inv_name} iter {it+1}/{ITERATIONS}: {elapsed:.4f}s')
            except Exception as e:
                print(f'  {inv_name} iter {it+1}/{ITERATIONS}: ERROR - {e}')
                success = False
                break

        if success and times:
            results.append({
                'task_id': task_id,
                'class': gt_class,
                'invariant': inv_name,
                'median_time': np.median(times),
                'mean_time': np.mean(times),
                'std_time': np.std(times),
                'min_time': np.min(times),
                'max_time': np.max(times),
                'iterations': len(times),
            })

print('\n=== Benchmark Complete ===')

In [ ]:
# === Results ===

df = pd.DataFrame(results)
print('\n--- Per-Sample Results ---')
print(df.to_string(index=False))

print('\n--- Summary by Invariant ---')
summary = df.groupby('invariant').agg(
    overall_median=('median_time', 'median'),
    overall_mean=('median_time', 'mean'),
    overall_std=('median_time', 'std'),
    overall_min=('median_time', 'min'),
    overall_max=('median_time', 'max'),
    n_samples=('median_time', 'count'),
).reset_index()
summary = summary.sort_values('overall_median')
print(summary.to_string(index=False))

# Save results
csv_path = 'benchmark_results.csv'
df.to_csv(csv_path, index=False)
print(f'\nResults saved to {csv_path}')

summary_path = 'benchmark_summary.csv'
summary.to_csv(summary_path, index=False)
print(f'Summary saved to {summary_path}')